<a href="https://colab.research.google.com/github/nikhilvirdi/Network-Intrusion-Detection-MLP-NSL-KDD/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Environment Check

Before writing any real code, confirm the Colab VM has the libraries we need
and record their exact versions. This matters because Colab's VM is fresh
every session and could, in principle, come with different pre-installed
versions — if something behaves unexpectedly later, version mismatch is the
first thing to rule out.

In [34]:
import sys
import torch
import pandas as pd
import sklearn
import numpy as np

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cpu
pandas: 2.2.2
NumPy: 2.0.2
scikit-learn: 1.6.1


### Step 2: Reproducibility (Random Seed)

Neural network training involves several sources of randomness — most
importantly, how a model's weights are initialized before training starts,
and how data gets shuffled. Without fixing this randomness, running the
exact same code twice can produce two different results, making it
impossible to tell whether a change in outcome came from an actual code
change or just random luck. Setting a fixed seed makes the "randomness"
reproducible — same seed, same sequence of "random" numbers, every run.

In [35]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### Step 3: Download the Dataset

The Colab VM we connected to is a fresh, empty machine — it has no files
on it except what we put there. NSL-KDD isn't bundled with PyTorch or
scikit-learn, so we need to fetch the raw train and test files from an
external source before pandas can read them. We'll download them directly
onto this VM's disk using a shell command, since Colab VM storage is
temporary and resets when the session ends — this download will need to be
redone if this runtime disconnects (e.g. after long inactivity).

In [36]:
!wget -q https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain+.txt
!wget -q https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest+.txt
!ls -la *.txt

-rw-r--r-- 1 root root  3441513 Aug  3 11:54 KDDTest+.txt
-rw-r--r-- 1 root root 19109424 Aug  3 11:54 KDDTrain+.txt


### Step 4: Load Data into pandas

These two files are plain CSV data, but critically, they have **no header
row** — the very first line is already a real data record, not column
names. If we load this naively, pandas will silently treat that first
record as the header, corrupting the entire dataset without throwing any
error. We'll fix this by explicitly supplying the 41 official NSL-KDD
feature names, plus two extra columns: `label` (the attack type, as text)
and `difficulty_level` (a score of how hard that record is to classify —
metadata about the record, not a real feature, and must never be fed into
the model as input).

In [37]:
column_names = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

train_df = pd.read_csv('KDDTrain+.txt', names=column_names)
test_df = pd.read_csv('KDDTest+.txt', names=column_names)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (125973, 43)
Test shape: (22544, 43)


### Step 5: Preview the Raw Data

Before deciding how to preprocess anything, we need to see what the data
actually looks like — real values, real types. This tells us which columns
are numeric (something a network can consume directly) versus categorical
text (something that needs converting first), and gives us a first
impression of the data's shape and quality.

In [38]:
train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [39]:
train_df.dtypes

,0
duration,int64
protocol_type,object
service,object
flag,object
src_bytes,int64
dst_bytes,int64
land,int64
wrong_fragment,int64
urgent,int64
hot,int64


### Step 6: Label Distribution

The `label` column holds the actual target we're trying to predict —
but it's not simply "normal" vs "attack." It contains dozens of specific
attack names. Before deciding how to frame this as a classification problem
(binary, or multi-class, or attack categories), we need to see exactly what
values exist and how skewed their counts are — this directly affects our
choice of loss function later.

In [40]:
train_df['label'].value_counts()

,count
label,
normal,67343
neptune,41214
satan,3633
ipsweep,3599
portsweep,2931
smurf,2646
nmap,1493
back,956
teardrop,892


### Step 7: Test Set Label Check

Before finalizing the 5-class framing, we need to confirm two things on
the test set: what labels actually appear there, and specifically which
attack types show up in test but were never seen during training. This is
the concrete generalization gap NSL-KDD is known for — a model that only
memorizes training patterns will struggle precisely on these unseen types.

In [41]:
train_labels = set(train_df['label'].unique())
test_labels = set(test_df['label'].unique())

print("Labels in train:", len(train_labels))
print("Labels in test:", len(test_labels))
print("\nIn test but NOT in train (unseen attacks):")
print(test_labels - train_labels)

Labels in train: 23
Labels in test: 38

In test but NOT in train (unseen attacks):
{'snmpgetattack', 'httptunnel', 'xterm', 'mscan', 'xlock', 'xsnoop', 'sqlattack', 'udpstorm', 'saint', 'processtable', 'snmpguess', 'sendmail', 'mailbomb', 'ps', 'named', 'apache2', 'worm'}


### Step 8: Check `worm` Occurrence Count

Before deciding how to fold `worm` into DoS or R2L, check how many test
rows it actually affects. This tells us whether this ambiguous call
meaningfully impacts evaluation, or is a negligible edge case.

In [42]:
worm_count = (test_df['label'] == 'worm').sum()
print("worm occurrences in test set:", worm_count)
print("as % of test set:", round(worm_count / len(test_df) * 100, 3), "%")

worm occurrences in test set: 2
as % of test set: 0.009 %


### Step 9: Problem Definition

**Task:** Multi-class classification of network connection records.

**Input:** 41 features per connection record (a mix of numeric measurements
and 3 categorical fields: `protocol_type`, `service`, `flag`).

**Output classes (5 total):**
- `normal` — legitimate traffic
- `DoS` (Denial of Service) — e.g. neptune, smurf, back, teardrop, pod,
  land, apache2, mailbomb, processtable, udpstorm, **worm**
- `Probe` — surveillance/scanning, e.g. satan, ipsweep, portsweep, nmap,
  mscan, saint
- `R2L` (Remote-to-Local) — unauthorized remote access, e.g. guess_passwd,
  warezclient, ftp_write, imap, phf, multihop, warezmaster, spy, sendmail,
  named, snmpgetattack, snmpguess, xlock, xsnoop, httptunnel
- `U2R` (User-to-Root) — privilege escalation, e.g. buffer_overflow,
  rootkit, loadmodule, perl, ps, sqlattack, xterm

**Known dataset characteristics to account for later:**
- Severe class imbalance even after 5-class grouping (`normal` and `DoS`
  dominate; `U2R` will remain the smallest by a wide margin) — affects our
  loss function choice (topic 3).
- Test set contains 17 attack types with zero training examples — a real,
  observable generalization gap (topic 4), not a hypothetical one.
- `difficulty_level` column is metadata, excluded from model input entirely.

**Model:** A PyTorch MLP (multi-layer perceptron) trained to take the 41
processed features as input and output a probability distribution over
the 5 classes.

### Step 10: Categorical Feature Inspection

We know `protocol_type`, `service`, and `flag` are text columns that need
converting to numbers before the model can use them. But *how* to convert
them depends on one thing we haven't checked yet: how many distinct values
each one has. A column with 3 possible values needs a different approach
than one with 70 — so we check cardinality (the number of unique values)
before choosing an encoding method.


In [43]:
for col in ['protocol_type', 'service', 'flag']:
    print(col, "->", train_df[col].nunique(), "unique values")

protocol_type -> 3 unique values
service -> 70 unique values
flag -> 11 unique values


### Step 11: Check Categorical Consistency Between Train and Test

Before encoding, verify whether any value in `protocol_type`, `service`,
or `flag` appears in the test set but not in train — the same kind of gap
we already found in the label column. If this exists, encoding train and
test separately would produce mismatched column counts, since one-hot
encoding only creates columns for values it has actually seen.

In [44]:
for col in ['protocol_type', 'service', 'flag']:
    train_vals = set(train_df[col].unique())
    test_vals = set(test_df[col].unique())
    print(col, "-> unseen in test:", test_vals - train_vals)

protocol_type -> unseen in test: set()
service -> unseen in test: set()
flag -> unseen in test: set()


### Step 12: One-Hot Encode Categorical Columns

We'll use scikit-learn's `OneHotEncoder`, which follows a two-stage
pattern used throughout sklearn: **fit**, then **transform**.

- `fit` means "look at this data and learn what categories exist" —
  it doesn't change any data, it just records the vocabulary.
- `transform` means "apply that already-learned encoding to this data."

Critically, we `fit` **only on the training set**, then use that same
learned encoding to `transform` both train and test. We do NOT let the
encoder look at test data while learning categories. Right now this
doesn't change the outcome, since we already confirmed test has no
categories missing from train — but it's a discipline worth building now,
because it becomes essential later with numeric scaling, where fitting on
test data would leak information about the test set into training and
quietly inflate our results. Better to build the habit here, where the
stakes are low, than get it wrong later where they aren't.

In [45]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['protocol_type', 'service', 'flag']

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(train_df[cat_cols])

train_encoded = encoder.transform(train_df[cat_cols])
test_encoded = encoder.transform(test_df[cat_cols])

print("Train encoded shape:", train_encoded.shape)
print("Test encoded shape:", test_encoded.shape)

Train encoded shape: (125973, 84)
Test encoded shape: (22544, 84)
